In [5]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

# --- Configuration ---
SDF_PATH = "Drugs/drugbank_approved_structures.sdf/structures.sdf"
MIN_DIAMETER_ANGSTROM = 20.0  # keep molecules with diameter > this value

def mol_diameter(mol):
    """Compute the maximum pairwise atom distance (diameter) in Ångströms.
    Uses existing 3D coords if present, otherwise generates them."""
    if mol.GetNumConformers() == 0:
        AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
    conf = mol.GetConformer()
    positions = conf.GetPositions()
    if np.allclose(positions[:, 2], 0):
        mol_3d = Chem.RWMol(mol)
        AllChem.EmbedMolecule(mol_3d, AllChem.ETKDGv3())
        if mol_3d.GetNumConformers() == 0:
            return None
        positions = mol_3d.GetConformer().GetPositions()
    diffs = positions[:, None, :] - positions[None, :, :]
    dists = np.sqrt((diffs ** 2).sum(axis=-1))
    return dists.max()

# --- Load and filter ---
supplier = Chem.ForwardSDMolSupplier(SDF_PATH, removeHs=False)
kept = []
skipped = 0

for mol in supplier:
    if mol is None:
        skipped += 1
        continue
    diameter = mol_diameter(mol)
    if diameter is not None and diameter > MIN_DIAMETER_ANGSTROM:
        mol.SetProp("MolDiameter", f"{diameter:.2f}")
        kept.append(mol)

print(f"Total kept: {len(kept)} molecules with diameter > {MIN_DIAMETER_ANGSTROM} Å")
print(f"Skipped (parse/embed failures): {skipped}")

# Show a summary of what was kept
print(f"\nTop 10:")
for mol in kept[:10]:
    name = mol.GetProp("GENERIC_NAME") if mol.HasProp("GENERIC_NAME") else "N/A"
    dbid = mol.GetProp("DRUGBANK_ID") if mol.HasProp("DRUGBANK_ID") else "N/A"
    diam = mol.GetProp("MolDiameter")
    print(f"  {dbid:10s}  {name:30s}  diameter = {diam} Å")

[22:38:31] Molecule does not have explicit Hs. Consider calling AddHs()
[22:38:31] Molecule does not have explicit Hs. Consider calling AddHs()
[22:38:31] Molecule does not have explicit Hs. Consider calling AddHs()
[22:38:31] Molecule does not have explicit Hs. Consider calling AddHs()
[22:38:32] Molecule does not have explicit Hs. Consider calling AddHs()
[22:38:32] Molecule does not have explicit Hs. Consider calling AddHs()
[22:38:33] Molecule does not have explicit Hs. Consider calling AddHs()
[22:38:33] Molecule does not have explicit Hs. Consider calling AddHs()
[22:38:34] Molecule does not have explicit Hs. Consider calling AddHs()
[22:38:34] Molecule does not have explicit Hs. Consider calling AddHs()
[22:38:34] UFFTYPER: Unrecognized atom type: Co5+3 (45)
[22:38:35] Molecule does not have explicit Hs. Consider calling AddHs()
[22:38:35] Molecule does not have explicit Hs. Consider calling AddHs()
[22:38:35] UFFTYPER: Unrecognized charge state for atom: 18
[22:38:35] Molecule 

Total kept: 87 molecules with diameter > 20.0 Å
Skipped (parse/embed failures): 2

Top 10:
  DB00035     Desmopressin                    diameter = 20.92 Å
  DB00115     Cyanocobalamin                  diameter = 23.65 Å
  DB00137     Lutein                          diameter = 27.66 Å
  DB00206     Reserpine                       diameter = 20.65 Å
  DB00390     Digoxin                         diameter = 24.01 Å
  DB00403     Ceruletide                      diameter = 33.56 Å
  DB00410     Mupirocin                       diameter = 20.50 Å
  DB00511     Acetyldigitoxin                 diameter = 24.92 Å
  DB00512     Vancomycin                      diameter = 23.02 Å
  DB00520     Caspofungin                     diameter = 25.45 Å


[22:45:46] Molecule does not have explicit Hs. Consider calling AddHs()


In [6]:
# --- Save the filtered molecules to a new SDF ---
OUTPUT_PATH = "Drugs/drugbank_filtered_large.sdf"

writer = Chem.SDWriter(OUTPUT_PATH)
for mol in kept:
    writer.write(mol)
writer.close()

print(f"Saved {len(kept)} molecules to {OUTPUT_PATH}")

Saved 87 molecules to Drugs/drugbank_filtered_large.sdf
